# FRED Data Explorer (Notebook)

A notebook version of the FRED Data Explorer app: fetch series from [FRED](https://fred.stlouisfed.org/), plot them over time, build derived series (ratios/differences), and plot relationship curves (Beveridge, Phillips) with NBER recession shading.

The setup cell below asks for your FRED API key interactively (via `getpass`, or `FRED_API_KEY` if already set as an environment variable) - no key is stored in this file, so it's safe to share.

## Setup

In [ ]:
# If needed, uncomment and run once:
# %pip install fredapi pandas matplotlib

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from fredapi import Fred

import getpass
import os

FRED_API_KEY = os.environ.get("FRED_API_KEY") or getpass.getpass(
    "Paste your FRED API key (free key: https://fred.stlouisfed.org/docs/api/api_key.html): "
)

fred = Fred(api_key=FRED_API_KEY)

## Helpers

Fetch multiple series into one aligned DataFrame, shade NBER recessions on a chart, and build a derived series from two others (e.g. a ratio).

In [ ]:
def fetch_series(series_map: dict, start: str = None, end: str = None) -> pd.DataFrame:
    """series_map: {series_id: label}. Returns one DataFrame aligned by date."""
    data = {}
    for series_id, label in series_map.items():
        s = fred.get_series(series_id, observation_start=start, observation_end=end)
        s.name = label
        data[label] = s
    return pd.concat(data.values(), axis=1)


_recession_cache = None


def get_recessions() -> pd.Series:
    global _recession_cache
    if _recession_cache is None:
        _recession_cache = fred.get_series("USREC")
    return _recession_cache


def shade_recessions(ax, recession_series: pd.Series = None):
    """Shade NBER recession periods on a matplotlib Axes with a date x-axis."""
    if recession_series is None:
        recession_series = get_recessions()
    xlim = ax.get_xlim()
    starts, ends = [], []
    in_rec = False
    for date, val in recession_series.items():
        if val == 1 and not in_rec:
            starts.append(date)
            in_rec = True
        elif val == 0 and in_rec:
            ends.append(date)
            in_rec = False
    if in_rec:
        ends.append(recession_series.index[-1])
    for s, e in zip(starts, ends):
        ax.axvspan(s, e, color="gray", alpha=0.15, zorder=0)
    ax.set_xlim(xlim)


def derived_series(df: pd.DataFrame, col_a: str, col_b: str, op: str = "ratio") -> pd.Series:
    """op: 'ratio' (A/B), 'difference' (A-B), 'sum' (A+B), 'product' (A*B), 'pct_of' (A/B*100)."""
    a, b = df[col_a], df[col_b]
    return {
        "ratio": a / b,
        "difference": a - b,
        "sum": a + b,
        "product": a * b,
        "pct_of": a / b * 100,
    }[op]

## Example 1 — Time series: Unemployment vs. Core PCE inflation

Two panels, each with a bold colored title, NBER recession shading, and a 10-year x-axis tick interval.

In [ ]:
START = "1950-01-01"

df = fetch_series({"UNRATE": "Unemployment Rate", "PCEPILFE": "Core PCE"}, start=START)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.5))

ax1.plot(df.index, df["Unemployment Rate"], color="#00205B", linewidth=1.5)
ax1.set_title("Unemployment Rate (%)", fontsize=10, fontweight="bold", color="#00205B")
ax1.xaxis.set_major_locator(mdates.YearLocator(10))
ax1.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax1.grid(alpha=0.3)
shade_recessions(ax1)

core_pce_yoy = df["Core PCE"].dropna().pct_change(12) * 100
ax2.plot(core_pce_yoy.index, core_pce_yoy.values, color="#c0392b", linewidth=1.5)
ax2.axhline(y=2, color="gray", linestyle="--", linewidth=0.8, label="2% target")
ax2.set_title("Core PCE Inflation (YoY %)", fontsize=10, fontweight="bold", color="#c0392b")
ax2.xaxis.set_major_locator(mdates.YearLocator(10))
ax2.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax2.legend(fontsize=7, loc="upper right")
ax2.grid(alpha=0.3)
shade_recessions(ax2)

fig.tight_layout()
plt.show()

## Example 2 — Derived series: V/E ratio (labor-market tightness)

Job Openings Rate ÷ Unemployment Rate.

In [ ]:
ve_df = fetch_series({"JTSJOR": "Job Openings Rate", "UNRATE": "Unemployment Rate"}, start="2000-01-01")
ve_ratio = derived_series(ve_df, "Job Openings Rate", "Unemployment Rate", op="ratio")

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.plot(ve_ratio.index, ve_ratio.values, color="#2CA02C", linewidth=1.5)
ax.set_title("V/E Ratio (Job Openings Rate / Unemployment Rate)", fontsize=10, fontweight="bold")
ax.grid(alpha=0.3)
shade_recessions(ax)
fig.tight_layout()
plt.show()

## Example 3 — Beveridge curve

Unemployment rate vs. job openings rate, points colored by year, latest point highlighted.

In [ ]:
bev = fetch_series({"UNRATE": "Unemployment Rate", "JTSJOR": "Job Openings Rate"}, start="2000-01-01").dropna()

fig, ax = plt.subplots(figsize=(6, 4.5))
sc = ax.scatter(
    bev["Unemployment Rate"], bev["Job Openings Rate"],
    c=bev.index.year, cmap="viridis", s=25, alpha=0.8, zorder=2,
)
ax.plot(bev["Unemployment Rate"], bev["Job Openings Rate"], color="gray", linewidth=0.5, alpha=0.4, zorder=1)

latest = bev.iloc[-1]
ax.scatter([latest["Unemployment Rate"]], [latest["Job Openings Rate"]], color="red", s=50, zorder=3)
ax.annotate(bev.index[-1].strftime("%Y-%m"), (latest["Unemployment Rate"], latest["Job Openings Rate"]),
            textcoords="offset points", xytext=(6, 6), fontsize=8, color="red")

fig.colorbar(sc, ax=ax, label="Year")
ax.set_title("Beveridge Curve", fontsize=11, fontweight="bold")
ax.set_xlabel("Unemployment Rate (%)")
ax.set_ylabel("Job Openings Rate (%)")
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

## Example 4 — Phillips curve

Unemployment rate vs. year-over-year CPI inflation.

In [ ]:
phillips_df = fetch_series({"UNRATE": "Unemployment Rate", "CPIAUCSL": "CPI"}, start="2000-01-01")
phillips_df["CPI YoY"] = phillips_df["CPI"].pct_change(12) * 100
phillips = phillips_df[["Unemployment Rate", "CPI YoY"]].dropna()

fig, ax = plt.subplots(figsize=(6, 4.5))
sc = ax.scatter(
    phillips["Unemployment Rate"], phillips["CPI YoY"],
    c=phillips.index.year, cmap="viridis", s=25, alpha=0.8, zorder=2,
)
ax.plot(phillips["Unemployment Rate"], phillips["CPI YoY"], color="gray", linewidth=0.5, alpha=0.4, zorder=1)

fig.colorbar(sc, ax=ax, label="Year")
ax.set_title("Phillips Curve", fontsize=11, fontweight="bold")
ax.set_xlabel("Unemployment Rate (%)")
ax.set_ylabel("CPI Inflation, YoY (%)")
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

## Customizing

- Swap series IDs in the `fetch_series({...})` calls — look up any FRED series ID at https://fred.stlouisfed.org/
- Change `START` / the `start=` argument to adjust the date range
- `derived_series(df, col_a, col_b, op=...)` supports `"ratio"`, `"difference"`, `"sum"`, `"product"`, `"pct_of"`
- Save any figure to a file with `fig.savefig("chart.png", dpi=200, bbox_inches="tight")`